In [1]:

from pathlib import Path
import json
import numpy as np

# ============================================================
# AVISTA REPRODUCIBILITY VALIDATION
# Step 1: Load exported data
# ============================================================

# Notebook is located beside the .avista project file
PROJECT_DIR = Path.cwd()
SPLIT_DIR = PROJECT_DIR / "outputs" / "data_split"

if not SPLIT_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {SPLIT_DIR}")

# Required train, validation, and test arrays
required_arrays = [
    "X_train", "X_val", "X_test",
    "y_train", "y_val", "y_test"
]

# Additional AVISTA arrays, loaded when available
optional_arrays = [
    "X_train_balanced",
    "y_train_balanced",
    "y_train_encoded",
    "y_val_encoded",
    "y_test_encoded",
    "y_train_balanced_encoded"
]

data = {}

for name in required_arrays + optional_arrays:
    file_path = SPLIT_DIR / f"{name}.npy"

    if not file_path.exists():
        if name in required_arrays:
            raise FileNotFoundError(f"Missing required file: {file_path}")
        continue

    data[name] = np.load(file_path, allow_pickle=True)

# Load split and imbalance metadata
def load_json(filename):
    path = SPLIT_DIR / filename
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

split_indices = load_json("split_indices.json")
imbalance_config = load_json("imbalance_config.json")

# Standard variables for subsequent notebook cells
X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]

# Display loaded data
print(f"Project: {PROJECT_DIR.name}")
print(f"Data directory: {SPLIT_DIR}")
print("-" * 65)

for name, array in data.items():
    print(f"{name:<28} Shape: {str(array.shape):<18} Dtype: {array.dtype}")

print("-" * 65)
print("Split indices available:", split_indices is not None)
print("Imbalance config available:", imbalance_config is not None)


Project: national-poll2
Data directory: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\outputs\data_split
-----------------------------------------------------------------
X_train                      Shape: (499, 60)          Dtype: float64
X_val                        Shape: (72, 60)           Dtype: float64
X_test                       Shape: (143, 60)          Dtype: float64
y_train                      Shape: (499,)             Dtype: object
y_val                        Shape: (72,)              Dtype: object
y_test                       Shape: (143,)             Dtype: object
X_train_balanced             Shape: (541, 60)          Dtype: float64
y_train_balanced             Shape: (541,)             Dtype: <U14
y_train_encoded              Shape: (499,)             Dtype: int64
y_val_encoded                Shape: (72,)              Dtype: int64
y_test_encoded               Shape: (143,)             Dtype: int64
y_train_balanced_e

In [2]:
# ============================================================
# Reproducibility results output directory
# ============================================================

REPRO_DIR = PROJECT_DIR / "reproduciablity_python"
REPRO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reproducibility outputs: {REPRO_DIR}")

Reproducibility outputs: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python


In [3]:

# ============================================================
# Step 2: Load AVISTA model configurations and metadata
# ============================================================

TRAINING_DIR = PROJECT_DIR / "outputs" / "training"

model_artifacts = {}

for config_path in sorted(TRAINING_DIR.glob("*/model_config.json")):
    model_name = config_path.parent.name
    model_dir = config_path.parent

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    metadata_path = model_dir / "training_metadata.json"

    metadata = {}
    if metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    model_artifacts[model_name] = {
        "directory": model_dir,
        "config": config,
        "metadata": metadata
    }

print(f"Found {len(model_artifacts)} trained model configurations")
print("=" * 70)

for model_name, artifacts in model_artifacts.items():
    print(f"\nMODEL: {model_name}")
    print("-" * 70)

    print("MODEL CONFIGURATION:")
    print(json.dumps(artifacts["config"], indent=2, default=str))

    print("\nTRAINING METADATA:")
    print(json.dumps(artifacts["metadata"], indent=2, default=str))

    print("=" * 70)

Found 5 trained model configurations

MODEL: FT-Transformer
----------------------------------------------------------------------
MODEL CONFIGURATION:
{
  "model_name": "ft_transformer",
  "display_name": "FT-Transformer",
  "input_dim": 60,
  "num_classes": 3,
  "n_features": 60,
  "n_classes": 3,
  "d_token": 128,
  "n_heads": 8,
  "n_layers": 3,
  "dropout": 0.1,
  "learning_rate": 0.001,
  "focal_gamma": 1.0,
  "cv_preprocessing_scope": "fold_training_only",
  "cv_imbalance_scope": "fold_training_only",
  "cv_source": "original_external_training_partition",
  "cv_resampling_before_split": false,
  "validation_role": "checkpoint_selection",
  "test_role": "final_evaluation",
  "cv_outer_validation_role": "fold_scoring_only",
  "cv_inner_validation_role": "early_stopping_when_feasible",
  "cv_inner_validation_fraction": 0.125,
  "cv_inner_random_seed": 42,
  "cv_inner_preprocessing_scope": "inner_training_only",
  "cv_inner_imbalance_scope": "inner_training_only",
  "cv_inner_valida

In [4]:

# ============================================================
# Step 3: Load AVISTA reference results for all models
# ============================================================

import json
import re
import pandas as pd

RESULTS_PATH = TRAINING_DIR / "training_results.json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    avista_run = json.load(f)


def normalize_model_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


# Index results by model name
avista_results = {
    normalize_model_name(item["model_name"]): item
    for item in avista_run["results"]
}

# Match result records with model configuration directories


def get_avista_reference(model_key):
    info = model_artifacts[model_key]

    possible_names = [
        info["config"].get("display_name"),
        model_key
    ]

    for name in possible_names:
        if name is None:
            continue

        key = normalize_model_name(name)

        if key in avista_results:
            return avista_results[key]

    raise KeyError(f"AVISTA result not found: {model_key}")


# Create automatic summary
summary_rows = []

for model_key in model_artifacts:
    result = get_avista_reference(model_key)

    test = result.get("test_metrics", {})
    cv = result.get("cv_summary", {})

    summary_rows.append({
        "Model": result["model_name"],
        "Test Accuracy": test.get("accuracy"),
        "Test Macro-F1": test.get("macro_f1"),
        "CV Accuracy Mean": cv.get("accuracy", {}).get("mean"),
        "CV Accuracy Std": cv.get("accuracy", {}).get("std"),
        "CV Macro-F1 Mean": cv.get("macro_f1", {}).get("mean"),
        "CV Macro-F1 Std": cv.get("macro_f1", {}).get("std")
    })

avista_summary_df = pd.DataFrame(summary_rows)

print(f"Project: {PROJECT_DIR.name}")
print(f"AVISTA version: {avista_run.get('application_version')}")
print(f"Loaded reference results: {len(avista_results)} models")
print("=" * 85)

display(avista_summary_df.round(6))

Project: national-poll2
AVISTA version: 1.1.0
Loaded reference results: 5 models


,Model,Test Accuracy,Test Macro-F1,CV Accuracy Mean,CV Accuracy Std,CV Macro-F1 Mean,CV Macro-F1 Std
0,FT-Transformer,0.335664,0.332862,0.336667,0.023190,0.319175,0.021557
1,Logistic Regression,0.398601,0.376359,0.374687,0.041221,0.363321,0.041105
2,Random Forest,0.433566,0.361977,0.374727,0.039061,0.305729,0.025252
3,TabPFN 2.5,0.517483,0.227343,0.521051,0.002101,0.228372,0.000604
4,XGBoost,0.475524,0.382068,0.412788,0.032616,0.306509,0.023687


In [5]:

# ============================================================
# Step 4: Logistic Regression — 5-fold CV
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Select model
MODEL_KEY = "LogisticRegression"

# Load saved settings
config = model_artifacts[MODEL_KEY]["config"]
reference = get_avista_reference(MODEL_KEY)

seed = int(config["random_state"])
n_folds = int(config["cv_folds"])

# Original exported training arrays
X_cv = data["X_train"]
y_cv = data["y_train_encoded"]

# Fresh Python estimator
params = config["parameters"].copy()
params["random_state"] = seed

model = LogisticRegression(**params)

# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=n_folds,
    shuffle=True,
    random_state=seed
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy"
}

if len(np.unique(y_cv)) == 2:
    scoring["roc_auc"] = "roc_auc"
else:
    scoring["roc_auc"] = "roc_auc_ovr"

cv_scores = cross_validate(
    model,
    X_cv,
    y_cv,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False
)

# Compare against AVISTA's saved CV results
rows = []

for metric in scoring:
    python_scores = cv_scores[f"test_{metric}"]

    python_mean = float(np.mean(python_scores))
    python_std = float(np.std(python_scores, ddof=0))

    avista_cv = reference.get("cv_summary", {}).get(metric, {})
    avista_mean = avista_cv.get("mean")
    avista_std = avista_cv.get("std")

    rows.append({
        "Metric": metric,
        "AVISTA Mean": avista_mean,
        "Python Mean": python_mean,
        "Mean Abs Diff": (
            abs(python_mean - avista_mean)
            if avista_mean is not None else np.nan
        ),
        "AVISTA Std": avista_std,
        "Python Std": python_std
    })

cv_comparison_df = pd.DataFrame(rows)

print(f"Dataset: {PROJECT_DIR.name}")
print(f"Model: {reference['model_name']}")
print(f"Random seed: {seed}")
print(f"CV folds: {n_folds}")
print("=" * 85)

display(cv_comparison_df.round(6))

# Individual Python fold results
fold_results_df = pd.DataFrame({
    "Fold": np.arange(1, n_folds + 1),
    **{
        metric: cv_scores[f"test_{metric}"]
        for metric in scoring
    }
})

print("\nIndividual Python fold results:")
display(fold_results_df.round(6))

Dataset: national-poll2
Model: Logistic Regression
Random seed: 42
CV folds: 5


,Metric,AVISTA Mean,Python Mean,Mean Abs Diff,AVISTA Std,Python Std
0,accuracy,0.374687,0.474929,0.100242,0.041221,0.032271
1,macro_f1,0.363321,0.298228,0.065093,0.041105,0.030791
2,weighted_f1,0.384955,0.401839,0.016885,0.038621,0.028928
3,balanced_accuracy,0.390551,0.333065,0.057485,0.053734,0.027710
4,roc_auc,0.534646,0.539535,0.004889,0.036855,0.040173



Individual Python fold results:


,Fold,accuracy,macro_f1,weighted_f1,balanced_accuracy,roc_auc
0,1,0.480000,0.305542,0.400423,0.338260,0.538460
1,2,0.530000,0.350201,0.450918,0.380057,0.590250
2,3,0.430000,0.256229,0.360606,0.294444,0.521091
3,4,0.470000,0.295608,0.403757,0.329487,0.475619
4,5,0.464646,0.283562,0.393492,0.323077,0.572256


In [6]:
lr_cv_path = REPRO_DIR / "LR_cv_comparison.csv"
lr_folds_path = REPRO_DIR / "LR_cv_fold_results.csv"

cv_comparison_df.to_csv(lr_cv_path, index=False)
fold_results_df.to_csv(lr_folds_path, index=False)

In [7]:

# ============================================================
# Step 5: Logistic Regression — Full Reproducibility Comparison
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Load model configuration and AVISTA reference results
# ------------------------------------------------------------

MODEL_KEY = "LogisticRegression"

config = model_artifacts[MODEL_KEY]["config"]
reference = get_avista_reference(MODEL_KEY)

seed = int(config["random_state"])
params = config["parameters"].copy()
params["random_state"] = seed

# ------------------------------------------------------------
# 2. Load exported training, validation, and test data
# ------------------------------------------------------------

X_fit = data.get("X_train_balanced", data["X_train"])

y_fit = data.get(
    "y_train_balanced_encoded",
    data["y_train_encoded"]
)

X_val = data["X_val"]
y_val = data["y_val_encoded"]

X_test = data["X_test"]
y_test = data["y_test_encoded"]

# ------------------------------------------------------------
# 3. Train a new model independently in Python
# ------------------------------------------------------------

lr_model = LogisticRegression(**params)
lr_model.fit(X_fit, y_fit)

# ------------------------------------------------------------
# 4. Calculate evaluation metrics
# ------------------------------------------------------------


def evaluate_classifier(model, X, y):
    pred = model.predict(X)

    metrics = {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(
            y, pred, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y, pred, average="weighted", zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_precision": precision_score(
            y, pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y, pred, average="macro", zero_division=0
        ),
    }

    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X)

            if len(model.classes_) == 2:
                metrics["roc_auc"] = roc_auc_score(
                    y, proba[:, 1]
                )
            else:
                metrics["roc_auc"] = roc_auc_score(
                    y,
                    proba,
                    labels=model.classes_,
                    multi_class="ovr",
                    average="macro"
                )
        except ValueError:
            metrics["roc_auc"] = np.nan

    return metrics, pred


# ------------------------------------------------------------
# 5. Compare with AVISTA reference metrics
# ------------------------------------------------------------

datasets = {
    "Train": (X_fit, y_fit),
    "Validation": (X_val, y_val),
    "Test": (X_test, y_test)
}

metric_names = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "roc_auc"
]

comparison_rows = []
python_predictions = {}

for split_name, (X, y) in datasets.items():

    python_metrics, predictions = evaluate_classifier(
        lr_model, X, y
    )

    python_predictions[split_name] = predictions

    avista_metrics = reference.get(
        f"{split_name.lower()}_metrics", {}
    )

    for metric in metric_names:

        avista_value = avista_metrics.get(metric, np.nan)
        python_value = python_metrics.get(metric, np.nan)

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": reference["model_name"],
            "Split": split_name,
            "Metric": metric,
            "AVISTA": avista_value,
            "Python": python_value,
            "Signed Difference": python_value - avista_value,
            "Absolute Difference": abs(
                python_value - avista_value
            )
        })

comparison_df = pd.DataFrame(comparison_rows)

# ------------------------------------------------------------
# 6. Display comparison by dataset partition
# ------------------------------------------------------------

print("=" * 85)
print("AVISTA vs. DIRECT PYTHON — LOGISTIC REGRESSION")
print("=" * 85)
print(f"Dataset:      {PROJECT_DIR.name}")
print(f"Model:        {reference['model_name']}")
print(f"Random seed:  {seed}")
print(f"CV folds:     {config['cv_folds']}")
print(f"Train rows:   {len(y_fit)}")
print(f"Val rows:     {len(y_val)}")
print(f"Test rows:    {len(y_test)}")

for split_name in datasets:

    print(f"\n{'=' * 85}")
    print(f"{split_name.upper()} METRICS")
    print("=" * 85)

    subset = comparison_df[
        comparison_df["Split"] == split_name
    ][[
        "Metric",
        "AVISTA",
        "Python",
        "Signed Difference",
        "Absolute Difference"
    ]]

    display(subset.round(6))

# ------------------------------------------------------------
# 7. Compare test-set confusion matrices
# ------------------------------------------------------------

print("\nTEST CONFUSION MATRICES")
print("=" * 85)

avista_cm = np.array(
    reference["test_metrics"]["confusion_matrix"]
)

python_cm = confusion_matrix(
    y_test,
    python_predictions["Test"],
    labels=lr_model.classes_
)

print("AVISTA:")
print(avista_cm)

print("\nDirect Python:")
print(python_cm)

print("\nNote: Matrix entries are directly comparable only")
print("after confirming identical class-label ordering.")

# ------------------------------------------------------------
# 8. Save reproducibility comparison
# ------------------------------------------------------------

# output_path = REPRO_DIR / "LR_reproducibility_results.csv"

# comparison_df.to_csv(output_path, index=False)

# print(f"\nComparison saved to: {output_path}")

AVISTA vs. DIRECT PYTHON — LOGISTIC REGRESSION
Dataset:      national-poll2
Model:        Logistic Regression
Random seed:  42
CV folds:     5
Train rows:   541
Val rows:     72
Test rows:    143

TRAIN METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
0,accuracy,0.517560,0.524954,0.007394,0.007394
1,macro_f1,0.516819,0.438549,-0.078270,0.078270
2,weighted_f1,0.519335,0.484300,-0.035035,0.035035
3,balanced_accuracy,0.544974,0.440151,-0.104823,0.104823
4,macro_precision,0.527317,0.524867,-0.002451,0.002451
5,macro_recall,0.544974,0.440151,-0.104823,0.104823
6,roc_auc,0.699686,0.701085,0.001399,0.001399



VALIDATION METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
7,accuracy,0.361111,0.486111,0.125000,0.125000
8,macro_f1,0.367735,0.359640,-0.008095,0.008095
9,weighted_f1,0.349822,0.440979,0.091157,0.091157
10,balanced_accuracy,0.409942,0.369006,-0.040936,0.040936
11,macro_precision,0.361270,0.395463,0.034193,0.034193
12,macro_recall,0.409942,0.369006,-0.040936,0.040936
13,roc_auc,0.575776,0.563583,-0.012193,0.012193



TEST METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
14,accuracy,0.398601,0.482517,0.083916,0.083916
15,macro_f1,0.376359,0.358824,-0.017536,0.017536
16,weighted_f1,0.405804,0.436185,0.030382,0.030382
17,balanced_accuracy,0.392096,0.369267,-0.022829,0.022829
18,macro_precision,0.388156,0.450294,0.062138,0.062138
19,macro_recall,0.392096,0.369267,-0.022829,0.022829
20,roc_auc,0.600150,0.601180,0.001030,0.001030



TEST CONFUSION MATRICES
AVISTA:
[[ 8 10 10]
 [15 28 31]
 [10 10 21]]

Direct Python:
[[ 3 20  5]
 [ 2 56 16]
 [ 1 30 10]]

Note: Matrix entries are directly comparable only
after confirming identical class-label ordering.


In [8]:

# ============================================================
# Logistic Regression — Diagnose Reproducibility Differences
# ============================================================

import joblib
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path

MODEL_KEY = "LogisticRegression"

model_dir = model_artifacts[MODEL_KEY]["directory"]
saved_path = model_dir / "trained_model.joblib"

if not saved_path.exists():
    print("Saved model not found.")
    print("Available files:")
    for p in model_dir.iterdir():
        print(" ", p.name)
else:
    # Only load trusted, locally generated AVISTA artifacts
    saved_object = joblib.load(saved_path)

    if isinstance(saved_object, dict):
        print("Saved object keys:", list(saved_object.keys()))
        avista_model = next(
            (
                saved_object[k]
                for k in ("model", "estimator", "pipeline")
                if k in saved_object
            ),
            saved_object
        )
    else:
        avista_model = saved_object

    print("AVISTA model type:", type(avista_model))
    print("Python model type:", type(lr_model))

    # Evaluate both models on identical processed test inputs
    X = data["X_test"]
    y = data["y_test_encoded"]

    pred_python = lr_model.predict(X)
    pred_avista = avista_model.predict(X)

    # Map original class labels to the exported encoded labels
    label_map = {}
    for raw, encoded in zip(
        data["y_train"], data["y_train_encoded"]
    ):
        key = str(raw)
        if key in label_map:
            assert label_map[key] == int(encoded)
        label_map[key] = int(encoded)

    def encode_predictions(pred):
        pred = np.asarray(pred).ravel()

        if all(
            isinstance(v, (int, np.integer))
            for v in pred
        ):
            return pred.astype(int)

        return np.array([
            label_map[str(v)] for v in pred
        ], dtype=int)

    pred_avista = encode_predictions(pred_avista)
    pred_python = encode_predictions(pred_python)

    print("\nTEST-SET COMPARISON")
    print("=" * 65)

    for name, pred in [
        ("AVISTA saved model", pred_avista),
        ("Independent Python", pred_python)
    ]:
        print(f"\n{name}")
        print(f"Accuracy: {accuracy_score(y, pred):.8f}")
        print(
            "Macro-F1: "
            f"{f1_score(y, pred, average='macro'):.8f}"
        )
        print(f"Incorrect predictions: {np.sum(y != pred)}")

    disagreement = pred_avista != pred_python

    print("\nPREDICTION-LEVEL COMPARISON")
    print("=" * 65)
    print(f"Test samples: {len(y)}")
    print(f"Disagreeing predictions: {disagreement.sum()}")
    print(f"Prediction agreement: {(~disagreement).mean():.6%}")

    print("\nMODEL PARAMETERS")
    print("=" * 65)

    avista_estimator = (
        avista_model.steps[-1][1]
        if hasattr(avista_model, "steps")
        else avista_model
    )

    if (
        hasattr(avista_estimator, "coef_")
        and hasattr(lr_model, "coef_")
    ):
        print(
            "Maximum coefficient difference:",
            np.max(
                np.abs(
                    avista_estimator.coef_ - lr_model.coef_
                )
            )
        )

        print(
            "Maximum intercept difference:",
            np.max(
                np.abs(
                    avista_estimator.intercept_ -
                    lr_model.intercept_
                )
            )
        )

        print("AVISTA iterations:", avista_estimator.n_iter_)
        print("Python iterations:", lr_model.n_iter_)

AVISTA model type: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Python model type: <class 'sklearn.linear_model._logistic.LogisticRegression'>

TEST-SET COMPARISON

AVISTA saved model
Accuracy: 0.39860140
Macro-F1: 0.37635949
Incorrect predictions: 86

Independent Python
Accuracy: 0.48251748
Macro-F1: 0.35882353
Incorrect predictions: 74

PREDICTION-LEVEL COMPARISON
Test samples: 143
Disagreeing predictions: 59
Prediction agreement: 58.741259%

MODEL PARAMETERS
Maximum coefficient difference: 0.10706232485688805
Maximum intercept difference: 0.33476862955973485
AVISTA iterations: [95]
Python iterations: [85]


c:\Users\monzu\anaconda3\envs\mimi\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [9]:

# ============================================================
# Compare AVISTA vs Python Logistic Regression parameters
# Compatible with cross-version saved models
# ============================================================

import numpy as np
import sklearn

print("ENVIRONMENT")
print("=" * 65)
print("Notebook sklearn:", sklearn.__version__)
print("AVISTA sklearn:  1.9.0 (from saved-model warning)")

# ------------------------------------------------------------
# 1. Verify training arrays
# ------------------------------------------------------------

print("\nTRAINING DATA CONSISTENCY")
print("=" * 65)

print(
    "X_train identical to X_train_balanced:",
    np.array_equal(
        data["X_train"],
        data["X_train_balanced"],
        equal_nan=True
    )
)

print(
    "y_train_encoded identical to y_train_balanced_encoded:",
    np.array_equal(
        data["y_train_encoded"],
        data["y_train_balanced_encoded"]
    )
)

# ------------------------------------------------------------
# 2. Compare parameters without AVISTA.get_params()
# ------------------------------------------------------------

print("\nMODEL PARAMETER COMPARISON")
print("=" * 65)

# Read saved AVISTA attributes directly
avista_state = vars(avista_model)

# Fresh Python estimator parameters
python_params = lr_model.get_params(deep=False)

differences = []
missing_attributes = []

for key, python_value in python_params.items():

    if key not in avista_state:
        missing_attributes.append(key)
        continue

    avista_value = avista_state[key]

    if repr(avista_value) != repr(python_value):
        differences.append({
            "Parameter": key,
            "AVISTA": repr(avista_value),
            "Python": repr(python_value)
        })

if differences:
    for item in differences:
        print(
            f"{item['Parameter']}: "
            f"AVISTA={item['AVISTA']}, "
            f"Python={item['Python']}"
        )
else:
    print("All shared estimator parameters match.")

if missing_attributes:
    print("\nAttributes absent from saved AVISTA model:")
    print(missing_attributes)

# ------------------------------------------------------------
# 3. Compare fitted coefficients and iterations
# ------------------------------------------------------------

print("\nFITTED MODEL DIFFERENCES")
print("=" * 65)

print(
    "Max coefficient difference:",
    np.max(np.abs(
        avista_model.coef_ - lr_model.coef_
    ))
)

print(
    "Max intercept difference:",
    np.max(np.abs(
        avista_model.intercept_ - lr_model.intercept_
    ))
)

print("AVISTA iterations:", avista_model.n_iter_)
print("Python iterations:", lr_model.n_iter_)

ENVIRONMENT
Notebook sklearn: 1.6.1
AVISTA sklearn:  1.9.0 (from saved-model warning)

TRAINING DATA CONSISTENCY
X_train identical to X_train_balanced: False
y_train_encoded identical to y_train_balanced_encoded: False

MODEL PARAMETER COMPARISON
class_weight: AVISTA={0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}, Python=None
random_state: AVISTA=None, Python=42

Attributes absent from saved AVISTA model:
['multi_class']

FITTED MODEL DIFFERENCES
Max coefficient difference: 0.10706232485688805
Max intercept difference: 0.33476862955973485
AVISTA iterations: [95]
Python iterations: [85]


In [10]:

# ============================================================
# Logistic Regression — Corrected Reproducibility Test
# Match AVISTA's effective class weights and random_state
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

MODEL_KEY = "LogisticRegression"

config = model_artifacts[MODEL_KEY]["config"]
reference = get_avista_reference(MODEL_KEY)

X_fit = data["X_train_balanced"]
y_fit = data["y_train_balanced_encoded"]

# ------------------------------------------------------------
# 1. Independently calculate balanced class weights
# ------------------------------------------------------------

classes = np.unique(y_fit)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_fit
)

class_weights = {
    int(c): float(w)
    for c, w in zip(classes, weights)
}

print("Calculated class weights:", class_weights)
print("AVISTA saved class weights:", avista_model.class_weight)

# ------------------------------------------------------------
# 2. Reconstruct AVISTA's effective estimator configuration
# ------------------------------------------------------------

params = config["parameters"].copy()

params["class_weight"] = class_weights

# Match actual saved estimator rather than assuming
# model_config.json contains every effective setting
params["random_state"] = avista_model.random_state

print("\nEffective Python parameters:")
print(params)

# ------------------------------------------------------------
# 3. Independent training
# ------------------------------------------------------------

lr_corrected = LogisticRegression(**params)
lr_corrected.fit(X_fit, y_fit)

# ------------------------------------------------------------
# 4. Compare with saved AVISTA reference metrics
# ------------------------------------------------------------

datasets = {
    "Train": (
        X_fit,
        y_fit
    ),
    "Validation": (
        data["X_val"],
        data["y_val_encoded"]
    ),
    "Test": (
        data["X_test"],
        data["y_test_encoded"]
    )
}

rows = []

for split_name, (X, y) in datasets.items():

    pred = lr_corrected.predict(X)

    python_metrics = {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(
            y, pred, average="macro"
        )
    }

    avista_metrics = reference[
        f"{split_name.lower()}_metrics"
    ]

    for metric, python_value in python_metrics.items():

        avista_value = avista_metrics[metric]

        rows.append({
            "Split": split_name,
            "Metric": metric,
            "AVISTA": avista_value,
            "Python Corrected": python_value,
            "Absolute Difference": abs(
                python_value - avista_value
            )
        })

corrected_df = pd.DataFrame(rows)

print("\n" + "=" * 75)
print("AVISTA vs. CORRECTED DIRECT PYTHON")
print("=" * 75)

display(corrected_df.round(8))

# ------------------------------------------------------------
# 5. Compare individual test predictions
# ------------------------------------------------------------

X_test = data["X_test"]
y_test = data["y_test_encoded"]

pred_avista = np.asarray(avista_model.predict(X_test))
pred_python = np.asarray(lr_corrected.predict(X_test))

disagreements = np.sum(pred_avista != pred_python)

print("\nPREDICTION AGREEMENT")
print("=" * 75)
print("Test observations:", len(y_test))
print("Disagreeing predictions:", disagreements)
print(
    "Prediction agreement:",
    f"{np.mean(pred_avista == pred_python):.6%}"
)

print("\nMODEL FIT COMPARISON")
print("=" * 75)

print(
    "Max coefficient difference:",
    np.max(
        np.abs(avista_model.coef_ - lr_corrected.coef_)
    )
)

print(
    "Max intercept difference:",
    np.max(
        np.abs(avista_model.intercept_ - lr_corrected.intercept_)
    )
)

print("AVISTA iterations:", avista_model.n_iter_)
print("Python iterations:", lr_corrected.n_iter_)

Calculated class weights: {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}
AVISTA saved class weights: {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}

Effective Python parameters:
{'penalty': 'l2', 'C': 1.0, 'l1_ratio': None, 'dual': False, 'tol': 0.0001, 'fit_intercept': True, 'intercept_scaling': 1.0, 'class_weight': {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}, 'solver': 'lbfgs', 'max_iter': 100, 'verbose': 0, 'warm_start': False, 'n_jobs': None, 'random_state': None}



AVISTA vs. CORRECTED DIRECT PYTHON


,Split,Metric,AVISTA,Python Corrected,Absolute Difference
0,Train,accuracy,0.517560,0.517560,0.0
1,Train,macro_f1,0.516819,0.516819,0.0
2,Validation,accuracy,0.361111,0.361111,0.0
3,Validation,macro_f1,0.367735,0.367735,0.0
4,Test,accuracy,0.398601,0.398601,0.0
5,Test,macro_f1,0.376359,0.376359,0.0



PREDICTION AGREEMENT
Test observations: 143
Disagreeing predictions: 0
Prediction agreement: 100.000000%

MODEL FIT COMPARISON
Max coefficient difference: 5.3607785045484135e-06
Max intercept difference: 6.804263462889537e-06
AVISTA iterations: [95]
Python iterations: [95]


In [11]:
# ============================================================
# SAVE CORRECTED LOGISTIC REGRESSION RESULTS
# ============================================================

REPRO_DIR = PROJECT_DIR / "reproduciablity_python"
REPRO_DIR.mkdir(parents=True, exist_ok=True)

# Add dataset and model identifiers
lr_final_df = corrected_df.copy()
lr_final_df.insert(0, "Dataset", PROJECT_DIR.name)
lr_final_df.insert(1, "Model", "Logistic Regression")

# Save corrected comparison
lr_output_path = REPRO_DIR / "LR_reproducibility_results.csv"

lr_final_df.to_csv(lr_output_path, index=False)

# Save prediction agreement and numerical diagnostics
lr_diagnostics = pd.DataFrame([{
    "Dataset": PROJECT_DIR.name,
    "Model": "Logistic Regression",
    "Test Samples": len(y_test),
    "Prediction Disagreements": disagreements,
    "Prediction Agreement": np.mean(pred_avista == pred_python),
    "Max Coefficient Difference": np.max(
        np.abs(avista_model.coef_ - lr_corrected.coef_)
    ),
    "Max Intercept Difference": np.max(
        np.abs(avista_model.intercept_ - lr_corrected.intercept_)
    ),
    "AVISTA Iterations": int(avista_model.n_iter_[0]),
    "Python Iterations": int(lr_corrected.n_iter_[0])
}])

diagnostic_path = REPRO_DIR / "LR_reproducibility_diagnostics.csv"
lr_diagnostics.to_csv(diagnostic_path, index=False)

print("CORRECTED LOGISTIC REGRESSION RESULTS SAVED")
print("=" * 65)
print(lr_output_path)
print(diagnostic_path)

CORRECTED LOGISTIC REGRESSION RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\LR_reproducibility_results.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\LR_reproducibility_diagnostics.csv


In [12]:

# ============================================================
# Step 6: Random Forest — Complete Reproducibility Validation
# AVISTA vs. Independent Python Implementation
# ============================================================

import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD MODEL CONFIGURATION AND AVISTA REFERENCE
# ============================================================

MODEL_KEY = "RandomForest"

config = model_artifacts[MODEL_KEY]["config"]
reference = get_avista_reference(MODEL_KEY)

seed = int(config["random_state"])
n_folds = int(config["cv_folds"])

params = config["parameters"].copy()
params["random_state"] = seed

print("=" * 85)
print("RANDOM FOREST — REPRODUCIBILITY VALIDATION")
print("=" * 85)
print(f"Dataset:         {PROJECT_DIR.name}")
print(f"Model:           {reference['model_name']}")
print(f"Random seed:     {seed}")
print(f"CV folds:        {n_folds}")
print(f"Scikit-learn:    {sklearn.__version__}")
print(f"AVISTA version:  {avista_run.get('application_version')}")

print("\nModel parameters:")
print(params)


# ============================================================
# 2. LOAD EXPORTED DATA
# ============================================================

# Model-fitting inputs
X_fit = data.get("X_train_balanced", data["X_train"])
y_fit = data.get(
    "y_train_balanced_encoded",
    data["y_train_encoded"]
)

# Validation and test inputs
X_val = data["X_val"]
y_val = data["y_val_encoded"]

X_test = data["X_test"]
y_test = data["y_test_encoded"]

# CV inputs: original, non-resampled exported training split
X_cv = data["X_train"]
y_cv = data["y_train_encoded"]

assert len(X_fit) == len(y_fit)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
assert len(X_cv) == len(y_cv)

print("\nData shapes:")
print(f"Training:   {X_fit.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"CV source:  {X_cv.shape}")


# ============================================================
# 3. INDEPENDENT FIVE-FOLD CROSS-VALIDATION
# ============================================================

print("\n" + "=" * 85)
print("PART A: FIVE-FOLD CROSS-VALIDATION")
print("=" * 85)

cv = StratifiedKFold(
    n_splits=n_folds,
    shuffle=True,
    random_state=seed
)

cv_model = RandomForestClassifier(**params)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy"
}

if len(np.unique(y_cv)) == 2:
    scoring["roc_auc"] = "roc_auc"
else:
    scoring["roc_auc"] = "roc_auc_ovr"

cv_scores = cross_validate(
    cv_model,
    X_cv,
    y_cv,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False,
    error_score="raise"
)

# Individual fold results
rf_fold_df = pd.DataFrame({
    "Fold": np.arange(1, n_folds + 1),
    **{
        metric: cv_scores[f"test_{metric}"]
        for metric in scoring
    }
})

print("\nIndividual Python fold results:")
display(rf_fold_df.round(6))

# Compare CV means and standard deviations
cv_rows = []

for metric in scoring:
    values = cv_scores[f"test_{metric}"]

    python_mean = float(np.mean(values))
    python_std = float(np.std(values, ddof=0))

    avista_values = reference.get(
        "cv_summary", {}
    ).get(metric, {})

    avista_mean = avista_values.get("mean")
    avista_std = avista_values.get("std")

    cv_rows.append({
        "Dataset": PROJECT_DIR.name,
        "Model": reference["model_name"],
        "Metric": metric,
        "AVISTA Mean": avista_mean,
        "Python Mean": python_mean,
        "Mean Abs Diff": (
            abs(python_mean - avista_mean)
            if avista_mean is not None else np.nan
        ),
        "AVISTA Std": avista_std,
        "Python Std": python_std
    })

rf_cv_comparison_df = pd.DataFrame(cv_rows)

print("\nAVISTA vs. Python — CV Comparison:")
display(rf_cv_comparison_df.round(6))


# ============================================================
# 4. TRAIN A FRESH RANDOM FOREST MODEL
# ============================================================

print("\n" + "=" * 85)
print("PART B: INDEPENDENT MODEL TRAINING")
print("=" * 85)

rf_model = RandomForestClassifier(**params)

rf_model.fit(X_fit, y_fit)

print("Independent Random Forest training completed.")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Number of features: {rf_model.n_features_in_}")
print(f"Classes: {rf_model.classes_}")


# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================

def evaluate_rf(model, X, y):

    predictions = model.predict(X)

    metrics = {
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(
            y, predictions,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y, predictions,
            average="weighted",
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y, predictions
        ),
        "macro_precision": precision_score(
            y, predictions,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y, predictions,
            average="macro",
            zero_division=0
        )
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)

        try:
            if len(model.classes_) == 2:
                metrics["roc_auc"] = roc_auc_score(
                    y, probabilities[:, 1]
                )
            else:
                metrics["roc_auc"] = roc_auc_score(
                    y,
                    probabilities,
                    labels=model.classes_,
                    multi_class="ovr",
                    average="macro"
                )
        except ValueError:
            metrics["roc_auc"] = np.nan

    return metrics, predictions


# ============================================================
# 6. TRAIN / VALIDATION / TEST COMPARISON
# ============================================================

print("\n" + "=" * 85)
print("PART C: AVISTA vs. PYTHON — FULL METRIC COMPARISON")
print("=" * 85)

datasets = {
    "Train": (X_fit, y_fit),
    "Validation": (X_val, y_val),
    "Test": (X_test, y_test)
}

metric_names = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "roc_auc"
]

comparison_rows = []
rf_predictions = {}

for split_name, (X, y) in datasets.items():

    python_metrics, predictions = evaluate_rf(
        rf_model, X, y
    )

    rf_predictions[split_name] = predictions

    avista_metrics = reference.get(
        f"{split_name.lower()}_metrics", {}
    )

    for metric in metric_names:

        avista_value = avista_metrics.get(
            metric, np.nan
        )

        python_value = python_metrics.get(
            metric, np.nan
        )

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": reference["model_name"],
            "Split": split_name,
            "Metric": metric,
            "AVISTA": avista_value,
            "Python": python_value,
            "Signed Difference": (
                python_value - avista_value
            ),
            "Absolute Difference": abs(
                python_value - avista_value
            )
        })

rf_comparison_df = pd.DataFrame(comparison_rows)

for split_name in datasets:

    print(f"\n{'=' * 85}")
    print(f"{split_name.upper()} METRICS")
    print("=" * 85)

    subset = rf_comparison_df[
        rf_comparison_df["Split"] == split_name
    ][[
        "Metric",
        "AVISTA",
        "Python",
        "Signed Difference",
        "Absolute Difference"
    ]]

    display(subset.round(6))


# ============================================================
# 7. TEST CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 85)
print("PART D: TEST CONFUSION MATRICES")
print("=" * 85)

avista_cm = np.asarray(
    reference["test_metrics"]["confusion_matrix"]
)

python_cm = confusion_matrix(
    y_test,
    rf_predictions["Test"],
    labels=rf_model.classes_
)

print("AVISTA confusion matrix:")
print(avista_cm)

print("\nPython confusion matrix:")
print(python_cm)

print(
    "\nNote: Elementwise comparison requires the same "
    "class-label ordering in both implementations."
)

if avista_cm.shape == python_cm.shape:
    print("\nNumerical matrix agreement (conditional on label order):")
    print(np.array_equal(avista_cm, python_cm))

print("\nPython classification report:")
print(
    classification_report(
        y_test,
        rf_predictions["Test"],
        zero_division=0
    )
)


# ============================================================
# 8. SAVE RESULTS
# ============================================================

rf_comparison_path = (
    REPRO_DIR / "RF_reproducibility_results.csv"
)

rf_cv_path = (
    REPRO_DIR / "RF_cv_comparison.csv"
)

rf_folds_path = (
    REPRO_DIR / "RF_cv_fold_results.csv"
)

rf_comparison_df.to_csv(
    rf_comparison_path, index=False
)

rf_cv_comparison_df.to_csv(
    rf_cv_path, index=False
)

rf_fold_df.to_csv(
    rf_folds_path, index=False
)

print("\n" + "=" * 85)
print("RESULTS SAVED")
print("=" * 85)

print(rf_comparison_path)
print(rf_cv_path)
print(rf_folds_path)

print("\nRandom Forest reproducibility experiment completed.")

RANDOM FOREST — REPRODUCIBILITY VALIDATION
Dataset:         national-poll2
Model:           Random Forest
Random seed:     42
CV folds:        5
Scikit-learn:    1.6.1
AVISTA version:  1.1.0

Model parameters:
{'n_estimators': 100, 'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'bootstrap': True, 'oob_score': False, 'n_jobs': None, 'verbose': 0, 'warm_start': False, 'class_weight': None, 'ccp_alpha': 0.0, 'max_samples': None, 'monotonic_cst': None, 'random_state': 42}

Data shapes:
Training:   (541, 60)
Validation: (72, 60)
Test:       (143, 60)
CV source:  (499, 60)

PART A: FIVE-FOLD CROSS-VALIDATION

Individual Python fold results:


,Fold,accuracy,macro_f1,weighted_f1,balanced_accuracy,roc_auc
0,1,0.390000,0.222003,0.328825,0.258685,0.424658
1,2,0.430000,0.316698,0.398222,0.330057,0.527762
2,3,0.550000,0.366939,0.486151,0.399573,0.495320
3,4,0.440000,0.304287,0.397495,0.329060,0.466620
4,5,0.393939,0.271728,0.365800,0.287607,0.475977



AVISTA vs. Python — CV Comparison:


,Dataset,Model,Metric,AVISTA Mean,Python Mean,Mean Abs Diff,AVISTA Std,Python Std
0,national-poll2,Random Forest,accuracy,0.374727,0.440788,0.066061,0.039061,0.057997
1,national-poll2,Random Forest,macro_f1,0.305729,0.296331,0.009398,0.025252,0.048153
2,national-poll2,Random Forest,weighted_f1,0.373937,0.395298,0.021361,0.034158,0.052070
3,national-poll2,Random Forest,balanced_accuracy,0.310004,0.320996,0.010992,0.027320,0.047579
4,national-poll2,Random Forest,roc_auc,0.483269,0.478067,0.005201,0.033596,0.033927



PART B: INDEPENDENT MODEL TRAINING
Independent Random Forest training completed.
Number of trees: 100
Number of features: 60
Classes: [0 1 2]

PART C: AVISTA vs. PYTHON — FULL METRIC COMPARISON

TRAIN METRICS


...


VALIDATION METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
7,accuracy,0.402778,0.402778,0.000000,0.000000
8,macro_f1,0.337217,0.235897,-0.101320,0.101320
9,weighted_f1,0.400717,0.343346,-0.057372,0.057372
10,balanced_accuracy,0.342690,0.271930,-0.070760,0.070760
11,macro_precision,0.341595,0.211478,-0.130117,0.130117
12,macro_recall,0.342690,0.271930,-0.070760,0.070760
13,roc_auc,0.595291,0.582196,-0.013095,0.013095



TEST METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
14,accuracy,0.433566,0.482517,0.048951,0.048951
15,macro_f1,0.361977,0.348416,-0.013560,0.013560
16,weighted_f1,0.424204,0.435609,0.011405,0.011405
17,balanced_accuracy,0.359787,0.365493,0.005705,0.005705
18,macro_precision,0.381403,0.379146,-0.002256,0.002256
19,macro_recall,0.359787,0.365493,0.005705,0.005705
20,roc_auc,0.545571,0.523578,-0.021993,0.021993



PART D: TEST CONFUSION MATRICES
AVISTA confusion matrix:
[[ 5 13 10]
 [ 4 45 25]
 [ 6 23 12]]

Python confusion matrix:
[[ 2 20  6]
 [ 3 56 15]
 [ 3 27 11]]

Note: Elementwise comparison requires the same class-label ordering in both implementations.

Numerical matrix agreement (conditional on label order):
False

Python classification report:
              precision    recall  f1-score   support

           0       0.25      0.07      0.11        28
           1       0.54      0.76      0.63        74
           2       0.34      0.27      0.30        41

    accuracy                           0.48       143
   macro avg       0.38      0.37      0.35       143
weighted avg       0.43      0.48      0.44       143


RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\RF_reproducibility_results.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\repro

In [13]:
# ============================================================
# 8. SAVE RANDOM FOREST RESULTS
# ============================================================

rf_comparison_path = REPRO_DIR / "RF_reproducibility_results.csv"
rf_cv_path = REPRO_DIR / "RF_cv_comparison.csv"
rf_folds_path = REPRO_DIR / "RF_cv_fold_results.csv"

rf_comparison_df.to_csv(rf_comparison_path, index=False)
rf_cv_comparison_df.to_csv(rf_cv_path, index=False)
rf_fold_df.to_csv(rf_folds_path, index=False)

print("\nRANDOM FOREST RESULTS SAVED")
print("=" * 65)
print(rf_comparison_path)
print(rf_cv_path)
print(rf_folds_path)


RANDOM FOREST RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\RF_reproducibility_results.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\RF_cv_comparison.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\RF_cv_fold_results.csv


In [14]:

# ============================================================
# Step 7: XGBoost — Complete Reproducibility Validation
# AVISTA vs. Independent Python Implementation
# ============================================================

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD MODEL CONFIGURATION AND AVISTA REFERENCE
# ============================================================

MODEL_KEY = "XGBoost"

config = model_artifacts[MODEL_KEY]["config"]
reference = get_avista_reference(MODEL_KEY)

seed = int(config["random_state"])
n_folds = int(config["cv_folds"])

params = config["parameters"].copy()
params["random_state"] = seed

print("=" * 85)
print("XGBOOST — REPRODUCIBILITY VALIDATION")
print("=" * 85)
print(f"Dataset:         {PROJECT_DIR.name}")
print(f"Model:           {reference['model_name']}")
print(f"Random seed:     {seed}")
print(f"CV folds:        {n_folds}")
print(f"Scikit-learn:    {sklearn.__version__}")
print(f"XGBoost:        {xgb.__version__}")
print(f"AVISTA version:  {avista_run.get('application_version')}")

print("\nModel parameters:")
print(params)


# ============================================================
# 2. LOAD EXPORTED DATA
# ============================================================

# Model-fitting inputs
X_fit = data.get("X_train_balanced", data["X_train"])

y_fit = data.get(
    "y_train_balanced_encoded",
    data["y_train_encoded"]
)

# Validation and test inputs
X_val = data["X_val"]
y_val = data["y_val_encoded"]

X_test = data["X_test"]
y_test = data["y_test_encoded"]

# Original exported training split for CV diagnostic
X_cv = data["X_train"]
y_cv = data["y_train_encoded"]

assert len(X_fit) == len(y_fit)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
assert len(X_cv) == len(y_cv)

print("\nData shapes:")
print(f"Training:   {X_fit.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"CV source:  {X_cv.shape}")


# ============================================================
# 3. FIVE-FOLD CROSS-VALIDATION
# ============================================================

print("\n" + "=" * 85)
print("PART A: FIVE-FOLD CROSS-VALIDATION")
print("=" * 85)

cv = StratifiedKFold(
    n_splits=n_folds,
    shuffle=True,
    random_state=seed
)

cv_model = XGBClassifier(**params)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy"
}

if len(np.unique(y_cv)) == 2:
    scoring["roc_auc"] = "roc_auc"
else:
    scoring["roc_auc"] = "roc_auc_ovr"

cv_scores = cross_validate(
    cv_model,
    X_cv,
    y_cv,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False,
    error_score="raise"
)

# Individual Python fold results
xgb_fold_df = pd.DataFrame({
    "Fold": np.arange(1, n_folds + 1),
    **{
        metric: cv_scores[f"test_{metric}"]
        for metric in scoring
    }
})

print("\nIndividual Python fold results:")
display(xgb_fold_df.round(6))


# Compare AVISTA and Python CV summaries
cv_rows = []

for metric in scoring:

    values = cv_scores[f"test_{metric}"]

    python_mean = float(np.mean(values))
    python_std = float(np.std(values, ddof=0))

    avista_values = reference.get(
        "cv_summary", {}
    ).get(metric, {})

    avista_mean = avista_values.get("mean")
    avista_std = avista_values.get("std")

    cv_rows.append({
        "Dataset": PROJECT_DIR.name,
        "Model": reference["model_name"],
        "Metric": metric,
        "AVISTA Mean": avista_mean,
        "Python Mean": python_mean,
        "Mean Abs Diff": (
            abs(python_mean - avista_mean)
            if avista_mean is not None else np.nan
        ),
        "AVISTA Std": avista_std,
        "Python Std": python_std
    })

xgb_cv_comparison_df = pd.DataFrame(cv_rows)

print("\nAVISTA vs. Python — CV Comparison:")
display(xgb_cv_comparison_df.round(6))


# ============================================================
# 4. TRAIN A FRESH XGBOOST MODEL
# ============================================================

print("\n" + "=" * 85)
print("PART B: INDEPENDENT MODEL TRAINING")
print("=" * 85)

xgb_model = XGBClassifier(**params)

xgb_model.fit(X_fit, y_fit)

print("Independent XGBoost training completed.")
print(f"Number of estimators: {xgb_model.n_estimators}")
print(f"Number of features: {xgb_model.n_features_in_}")
print(f"Classes: {xgb_model.classes_}")


# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================

def evaluate_xgb(model, X, y):

    predictions = model.predict(X)

    metrics = {
        "accuracy": accuracy_score(y, predictions),

        "macro_f1": f1_score(
            y, predictions,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y, predictions,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y, predictions
        ),

        "macro_precision": precision_score(
            y, predictions,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y, predictions,
            average="macro",
            zero_division=0
        )
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)

        try:
            if len(model.classes_) == 2:
                metrics["roc_auc"] = roc_auc_score(
                    y, probabilities[:, 1]
                )
            else:
                metrics["roc_auc"] = roc_auc_score(
                    y,
                    probabilities,
                    labels=model.classes_,
                    multi_class="ovr",
                    average="macro"
                )
        except ValueError:
            metrics["roc_auc"] = np.nan

    return metrics, predictions


# ============================================================
# 6. TRAIN / VALIDATION / TEST COMPARISON
# ============================================================

print("\n" + "=" * 85)
print("PART C: AVISTA vs. PYTHON — METRIC COMPARISON")
print("=" * 85)

datasets = {
    "Train": (X_fit, y_fit),
    "Validation": (X_val, y_val),
    "Test": (X_test, y_test)
}

metric_names = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "roc_auc"
]

comparison_rows = []
xgb_predictions = {}

for split_name, (X, y) in datasets.items():

    python_metrics, predictions = evaluate_xgb(
        xgb_model, X, y
    )

    xgb_predictions[split_name] = predictions

    avista_metrics = reference.get(
        f"{split_name.lower()}_metrics", {}
    )

    for metric in metric_names:

        avista_value = avista_metrics.get(
            metric, np.nan
        )

        python_value = python_metrics.get(
            metric, np.nan
        )

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": reference["model_name"],
            "Split": split_name,
            "Metric": metric,
            "AVISTA": avista_value,
            "Python": python_value,
            "Signed Difference": (
                python_value - avista_value
            ),
            "Absolute Difference": abs(
                python_value - avista_value
            )
        })

xgb_comparison_df = pd.DataFrame(comparison_rows)

for split_name in datasets:

    print(f"\n{'=' * 85}")
    print(f"{split_name.upper()} METRICS")
    print("=" * 85)

    subset = xgb_comparison_df[
        xgb_comparison_df["Split"] == split_name
    ][[
        "Metric",
        "AVISTA",
        "Python",
        "Signed Difference",
        "Absolute Difference"
    ]]

    display(subset.round(6))


# ============================================================
# 7. TEST CONFUSION MATRICES AND PREDICTION AGREEMENT
# ============================================================

print("\n" + "=" * 85)
print("PART D: TEST CONFUSION MATRICES")
print("=" * 85)

avista_cm = np.asarray(
    reference["test_metrics"]["confusion_matrix"]
)

python_cm = confusion_matrix(
    y_test,
    xgb_predictions["Test"],
    labels=xgb_model.classes_
)

print("AVISTA confusion matrix:")
print(avista_cm)

print("\nPython confusion matrix:")
print(python_cm)

# Class ordering must be verified before interpreting
# elementwise matrix agreement.
print("\nConfusion matrices numerically identical:",
      np.array_equal(avista_cm, python_cm))

print("\nPython classification report:")
print(
    classification_report(
        y_test,
        xgb_predictions["Test"],
        zero_division=0
    )
)

# Compare exact metrics without relying on rounded values
avista_test = reference["test_metrics"]

print("\nTEST METRIC DIFFERENCES")
print("=" * 85)

for metric in ["accuracy", "macro_f1"]:
    row = xgb_comparison_df[
        (xgb_comparison_df["Split"] == "Test") &
        (xgb_comparison_df["Metric"] == metric)
    ].iloc[0]

    print(
        f"{metric:<18} "
        f"AVISTA={row['AVISTA']:.8f}  "
        f"Python={row['Python']:.8f}  "
        f"Difference={row['Absolute Difference']:.8f}"
    )


# ============================================================
# 8. SAVE RESULTS
# ============================================================

REPRO_DIR.mkdir(parents=True, exist_ok=True)

xgb_comparison_path = (
    REPRO_DIR / "XGB_reproducibility_results.csv"
)

xgb_cv_path = (
    REPRO_DIR / "XGB_cv_comparison.csv"
)

xgb_folds_path = (
    REPRO_DIR / "XGB_cv_fold_results.csv"
)

xgb_comparison_df.to_csv(
    xgb_comparison_path, index=False
)

xgb_cv_comparison_df.to_csv(
    xgb_cv_path, index=False
)

xgb_fold_df.to_csv(
    xgb_folds_path, index=False
)

print("\n" + "=" * 85)
print("RESULTS SAVED")
print("=" * 85)

print(xgb_comparison_path)
print(xgb_cv_path)
print(xgb_folds_path)

print("\nXGBoost reproducibility experiment completed.")

XGBOOST — REPRODUCIBILITY VALIDATION
Dataset:         national-poll2
Model:           XGBoost
Random seed:     42
CV folds:        5
Scikit-learn:    1.6.1
XGBoost:        3.0.2
AVISTA version:  1.1.0

Model parameters:
{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'tree_method': 'auto', 'booster': 'gbtree', 'gamma': 0.0, 'min_child_weight': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'scale_pos_weight': 1.0, 'random_state': 42, 'n_jobs': None, 'verbosity': 0, 'enable_categorical': False}

Data shapes:
Training:   (541, 60)
Validation: (72, 60)
Test:       (143, 60)
CV source:  (499, 60)

PART A: FIVE-FOLD CROSS-VALIDATION

Individual Python fold results:


,Fold,accuracy,macro_f1,weighted_f1,balanced_accuracy,roc_auc
0,1,0.440000,0.338670,0.420825,0.343016,0.501435
1,2,0.390000,0.301769,0.376344,0.309117,0.510313
2,3,0.480000,0.335938,0.439151,0.359402,0.490781
3,4,0.390000,0.300648,0.371694,0.309117,0.440493
4,5,0.424242,0.302466,0.398641,0.316239,0.492486



AVISTA vs. Python — CV Comparison:


,Dataset,Model,Metric,AVISTA Mean,Python Mean,Mean Abs Diff,AVISTA Std,Python Std
0,national-poll2,XGBoost,accuracy,0.412788,0.424848,0.012061,0.032616,0.033765
1,national-poll2,XGBoost,macro_f1,0.306509,0.315898,0.009389,0.023687,0.017509
2,national-poll2,XGBoost,weighted_f1,0.389932,0.401331,0.011399,0.027171,0.025769
3,national-poll2,XGBoost,balanced_accuracy,0.315959,0.327378,0.011419,0.023738,0.020303
4,national-poll2,XGBoost,roc_auc,0.485602,0.487102,0.001499,0.026227,0.024327



PART B: INDEPENDENT MODEL TRAINING
Independent XGBoost training completed.
Number of estimators: 300
Number of features: 60
Classes: [0 1 2]

PART C: AVISTA vs. PYTHON — METRIC COMPARISON

TRAIN METRICS


...


VALIDATION METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
7,accuracy,0.416667,0.388889,-0.027778,0.027778
8,macro_f1,0.281661,0.268111,-0.013550,0.013550
9,weighted_f1,0.374642,0.355065,-0.019577,0.019577
10,balanced_accuracy,0.307018,0.289474,-0.017544,0.017544
11,macro_precision,0.262661,0.250840,-0.011821,0.011821
12,macro_recall,0.307018,0.289474,-0.017544,0.017544
13,roc_auc,0.539835,0.550907,0.011073,0.011073



TEST METRICS


,Metric,AVISTA,Python,Signed Difference,Absolute Difference
14,accuracy,0.475524,0.475524,0.000000,0.000000
15,macro_f1,0.382068,0.390165,0.008097,0.008097
16,weighted_f1,0.450228,0.454331,0.004103,0.004103
17,balanced_accuracy,0.383040,0.386815,0.003775,0.003775
18,macro_precision,0.452224,0.448718,-0.003507,0.003507
19,macro_recall,0.383040,0.386815,0.003775,0.003775
20,roc_auc,0.558880,0.558966,0.000086,0.000086



PART D: TEST CONFUSION MATRICES
AVISTA confusion matrix:
[[ 4 15  9]
 [ 2 51 21]
 [ 2 26 13]]

Python confusion matrix:
[[ 5 13 10]
 [ 3 51 20]
 [ 2 27 12]]

Confusion matrices numerically identical: False

Python classification report:
              precision    recall  f1-score   support

           0       0.50      0.18      0.26        28
           1       0.56      0.69      0.62        74
           2       0.29      0.29      0.29        41

    accuracy                           0.48       143
   macro avg       0.45      0.39      0.39       143
weighted avg       0.47      0.48      0.45       143


TEST METRIC DIFFERENCES
accuracy           AVISTA=0.47552448  Python=0.47552448  Difference=0.00000000
macro_f1           AVISTA=0.38206795  Python=0.39016545  Difference=0.00809749

RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python\XGB_reproducibility_results.csv
d:\TXST\OneDrive - Texas Stat

In [15]:
# ============================================================
# FT-Transformer: Direct PyTorch reproduction (AVISTA exported inputs)
# Requires prior notebook cells: PROJECT_DIR, data, model_artifacts,
# get_avista_reference; outputs are stored in reproduciablity_python.
# ============================================================
from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, roc_auc_score)

FT_MODEL_KEY = 'FT-Transformer'
ft_config = model_artifacts[FT_MODEL_KEY]['config']
ft_reference = get_avista_reference(FT_MODEL_KEY)
ft_seed = int(ft_config.get('cv_inner_random_seed', 42))
ft_folds = 5
ft_metadata = model_artifacts[FT_MODEL_KEY]['metadata']
ft_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ft_outdir = PROJECT_DIR / 'reproduciablity_python'
ft_outdir.mkdir(parents=True, exist_ok=True)

# AVISTA's trainer.py does not seed PyTorch within fit_model(). Upstream
# seeding may exist; we seed this standalone test for repeatable runs.
torch.manual_seed(ft_seed)
np.random.seed(ft_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(ft_seed)

ft_x_train = np.asarray(data['X_train_balanced'], dtype=np.float32)
ft_y_train = np.asarray(data['y_train_balanced_encoded'], dtype=np.int64)
ft_x_val = np.asarray(data['X_val'], dtype=np.float32)
ft_y_val = np.asarray(data['y_val_encoded'], dtype=np.int64)
ft_x_test = np.asarray(data['X_test'], dtype=np.float32)
ft_y_test = np.asarray(data['y_test_encoded'], dtype=np.int64)
ft_classes = np.unique(np.concatenate([ft_y_train, ft_y_val, ft_y_test]))
assert np.array_equal(ft_classes, np.arange(
    len(ft_classes))), 'Target labels must be 0..K-1'

# AVISTA trainer.py defaults not fully preserved in model_config.json
ft_batch_size = 128
ft_max_epochs = 80
ft_warmup_epochs = 5
ft_patience = 30
ft_weight_decay = 1e-3
ft_label_smoothing = 0.05
ft_gamma = float(ft_config.get('focal_gamma', 1.0))
ft_lr = float(ft_config.get('learning_rate', 0.001))

# Architecture matched to user's torch_tabular_models.py (separate class here).


class ReferenceFTTransformer(nn.Module):
    def __init__(self, n_features, n_classes, d_token=128, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.kaiming_uniform_(self.W, a=0.01)
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        enc = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(nn.Linear(d_token, d_token // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_token // 2, n_classes))

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)
        cls = self.cls.expand(x.shape[0], -1, -1)
        return self.head(self.norm(self.enc(torch.cat([cls, tokens], dim=1))[:, 0]))


def ft_loader(X, y, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32),
                                    torch.tensor(y, dtype=torch.long)),
                      batch_size=ft_batch_size, shuffle=shuffle)


def ft_evaluate(model, X, y):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch, _ in ft_loader(X, np.zeros(len(X), dtype=np.int64)):
            probs.append(torch.softmax(
                model(batch.to(ft_device)), dim=1).cpu().numpy())
    probs = np.vstack(probs)
    pred = probs.argmax(axis=1)
    metrics = {
        'accuracy': accuracy_score(y, pred),
        'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y, pred, average='weighted', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
        'macro_precision': precision_score(y, pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y, pred, average='macro', zero_division=0)
    }
    try:
        metrics['roc_auc'] = (roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2
                              else roc_auc_score(y, probs, multi_class='ovr', average='macro'))
    except ValueError:
        metrics['roc_auc'] = np.nan
    return metrics, pred


# Reproduces trainer.py's PyTorch training semantics (not identical random draws).
def ft_fit(X, y, Xvalid, yvalid):
    model = ReferenceFTTransformer(
        n_features=X.shape[1], n_classes=len(ft_classes),
        d_token=int(ft_config['d_token']), n_heads=int(ft_config['n_heads']),
        n_layers=int(ft_config['n_layers']), dropout=float(ft_config['dropout'])
    ).to(ft_device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=ft_lr, weight_decay=ft_weight_decay)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, ft_max_epochs - ft_warmup_epochs))
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=max(1, ft_warmup_epochs))
    weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    alpha = torch.ones(len(ft_classes), dtype=torch.float32, device=ft_device)
    for klass, weight in zip(np.unique(y), weights):
        alpha[int(klass)] = float(weight)
    loader = ft_loader(X, y, shuffle=True)
    validation = Xvalid is not None and len(yvalid) > 0
    best_f1, best_epoch, wait, best_state = -1.0, None, 0, None
    for epoch in range(1, ft_max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(ft_device), yb.to(ft_device)
            optimizer.zero_grad()
            logits = model(xb)
            ce = F.cross_entropy(logits, yb, weight=alpha,
                                 label_smoothing=ft_label_smoothing, reduction='none')
            loss = ((1.0 - torch.exp(-ce)) ** ft_gamma * ce).mean()
            loss.backward()
            optimizer.step()
        if validation:
            validation_metrics, _ = ft_evaluate(model, Xvalid, yvalid)
            val_f1 = validation_metrics['macro_f1']
            if val_f1 > best_f1:
                best_f1, best_epoch, wait = val_f1, epoch, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
        if epoch <= ft_warmup_epochs:
            warmup.step()
        else:
            cosine.step()
        if validation and wait >= ft_patience:
            break
    if validation and best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, epoch


print('FT-Transformer |', PROJECT_DIR.name, '| device:', ft_device,
      '| torch:', torch.__version__, '| seed:', ft_seed)
ft_model, ft_best_epoch, ft_epochs_trained = ft_fit(
    ft_x_train, ft_y_train, ft_x_val, ft_y_val)
ft_rows = []
for split, (X, y) in {'Train': (ft_x_train, ft_y_train),
                      'Validation': (ft_x_val, ft_y_val),
                      'Test': (ft_x_test, ft_y_test)}.items():
    metrics, _ = ft_evaluate(ft_model, X, y)
    ref = ft_reference.get(split.lower() + '_metrics', {})
    for metric, value in metrics.items():
        av = ref.get(metric, np.nan)
        ft_rows.append({'Dataset': PROJECT_DIR.name, 'Model': 'FT-Transformer',
                        'Split': split, 'Metric': metric, 'AVISTA': av,
                        'Python': value, 'Absolute Difference': abs(value - av)})
ft_compare_df = pd.DataFrame(ft_rows)
display(ft_compare_df.round(6))
print('AVISTA best epoch:', ft_metadata.get('best_epoch'),
      '| Python best epoch:', ft_best_epoch, '| Python epochs trained:', ft_epochs_trained)
ft_compare_df.to_csv(ft_outdir / 'FT_reproducibility_results.csv', index=False)

# Note: this CV is a DIAGNOSTIC on exported prepared features. AVISTA
# independently fits preprocessing/resampling inside the inner training fold.
# It is NOT a reproduction of that full fold-local CV pipeline.
FT_RUN_CV_DIAGNOSTIC = True
if FT_RUN_CV_DIAGNOSTIC:
    cv = StratifiedKFold(n_splits=ft_folds, shuffle=True, random_state=ft_seed)
    ft_cv_rows = []
    Xcv = np.asarray(data['X_train'], dtype=np.float32)
    ycv = np.asarray(data['y_train_encoded'], dtype=np.int64)
    fraction = float(ft_config.get('cv_inner_validation_fraction', 0.125))
    for fold, (outer_train, outer_eval) in enumerate(cv.split(Xcv, ycv), start=1):
        inner_train, inner_val = train_test_split(
            outer_train, test_size=fraction, stratify=ycv[outer_train], random_state=ft_seed)
        fmodel, best, trained = ft_fit(Xcv[inner_train], ycv[inner_train],
                                       Xcv[inner_val], ycv[inner_val])
        fm, _ = ft_evaluate(fmodel, Xcv[outer_eval], ycv[outer_eval])
        ft_cv_rows.append({'Fold': fold, 'Best epoch': best,
                          'Epochs trained': trained, **fm})
        print('FT CV diagnostic fold', fold, 'of', ft_folds,
              'macro-F1:', round(fm['macro_f1'], 6))
        del fmodel
        if ft_device.type == 'cuda':
            torch.cuda.empty_cache()
    ft_cv_folds_df = pd.DataFrame(ft_cv_rows)
    ft_cv_folds_df.to_csv(
        ft_outdir / 'FT_cv_fold_results_DIAGNOSTIC.csv', index=False)
    ft_cv_comp = []
    for metric in ('accuracy', 'macro_f1', 'weighted_f1', 'balanced_accuracy', 'roc_auc'):
        ref = ft_reference.get('cv_summary', {}).get(metric, {})
        ft_cv_comp.append({'Metric': metric, 'AVISTA Mean': ref.get('mean', np.nan),
                           'Python Mean': ft_cv_folds_df[metric].mean(),
                           'AVISTA Std': ref.get('std', np.nan),
                           'Python Std': ft_cv_folds_df[metric].std(ddof=0),
                           'Protocol': 'exported-feature CV diagnostic, not fold-local pipeline parity'})
    ft_cv_comparison_df = pd.DataFrame(ft_cv_comp)
    display(ft_cv_comparison_df.round(6))
    ft_cv_comparison_df.to_csv(
        ft_outdir / 'FT_cv_comparison_DIAGNOSTIC.csv', index=False)
print('FT outputs saved to:', ft_outdir)

FT-Transformer | national-poll2 | device: cuda | torch: 2.6.0+cu118 | seed: 42


c:\Users\monzu\anaconda3\envs\mimi\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


,Dataset,Model,Split,Metric,AVISTA,Python,Absolute Difference
0,national-poll2,FT-Transformer,Train,accuracy,0.495379,0.395564,0.099815
1,national-poll2,FT-Transformer,Train,macro_f1,0.496033,0.376591,0.119442
2,national-poll2,FT-Transformer,Train,weighted_f1,0.476263,0.330321,0.145942
3,national-poll2,FT-Transformer,Train,balanced_accuracy,0.559433,0.475301,0.084131
4,national-poll2,FT-Transformer,Train,macro_precision,0.530125,0.437119,0.093007
5,national-poll2,FT-Transformer,Train,macro_recall,0.559433,0.475301,0.084131
6,national-poll2,FT-Transformer,Train,roc_auc,0.767658,0.676278,0.091380
7,national-poll2,FT-Transformer,Validation,accuracy,0.375000,0.361111,0.013889
8,national-poll2,FT-Transformer,Validation,macro_f1,0.372173,0.359712,0.012461
9,national-poll2,FT-Transformer,Validation,weighted_f1,0.373679,0.305542,0.068138


AVISTA best epoch: 33 | Python best epoch: 18 | Python epochs trained: 48


c:\Users\monzu\anaconda3\envs\mimi\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


...

FT CV diagnostic fold 4 of 5 macro-F1: 0.284848


c:\Users\monzu\anaconda3\envs\mimi\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


FT CV diagnostic fold 5 of 5 macro-F1: 0.316238


,Metric,AVISTA Mean,Python Mean,AVISTA Std,Python Std,Protocol
0,accuracy,0.336667,0.332646,0.023190,0.029020,"exported-feature CV diagnostic, not fold-local..."
1,macro_f1,0.319175,0.296379,0.021557,0.041498,"exported-feature CV diagnostic, not fold-local..."
2,weighted_f1,0.333094,0.324443,0.025613,0.055201,"exported-feature CV diagnostic, not fold-local..."
3,balanced_accuracy,0.353074,0.327263,0.016975,0.024753,"exported-feature CV diagnostic, not fold-local..."
4,roc_auc,0.529781,0.521291,0.019192,0.035441,"exported-feature CV diagnostic, not fold-local..."


FT outputs saved to: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python


In [16]:
%pip install tabpfn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import sys
import tabpfn
import torch
from importlib.metadata import version

print("Python:", sys.executable)
print("TabPFN:", version("tabpfn"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

Python: c:\Users\monzu\anaconda3\envs\mimi\python.exe
TabPFN: 9.2.0
PyTorch: 2.6.0+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 2070 SUPER


c:\Users\monzu\anaconda3\envs\mimi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
# ============================================================
# TabPFN 2.5: Direct Python reproduction with native raw inputs
# Requires prior notebook cells: PROJECT_DIR, data, model_artifacts,
# get_avista_reference; saves to reproduciablity_python.
# ============================================================
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import tabpfn
from tabpfn import TabPFNClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, roc_auc_score)

TP_KEY = 'TabPFN_2_5'
tp_config = model_artifacts[TP_KEY]['config']
tp_metadata = model_artifacts[TP_KEY]['metadata']
tp_reference = get_avista_reference(TP_KEY)
tp_seed = int(tp_metadata.get('training_subsampling_seed', 42))
tp_folds = 5
tp_output = PROJECT_DIR / 'reproduciablity_python'
tp_output.mkdir(parents=True, exist_ok=True)

# trainer.py reloads the original project dataset, or data/modeling_subset.csv.
# The fallback here uses a local modeling subset. If that CSV is not present,
# provide the actual original project dataset path instead of guessing.
tp_raw_path = PROJECT_DIR / 'data' / 'modeling_subset.csv'
if not tp_raw_path.exists():
    raise FileNotFoundError(
        f'Raw modeling data missing: {tp_raw_path}. '
        'Specify the actual AVISTA project dataset path for TabPFN.')
tp_raw = pd.read_csv(tp_raw_path)
tp_indices = split_indices
if tp_indices is None:
    with open(PROJECT_DIR / 'outputs' / 'data_split' / 'split_indices.json', encoding='utf-8') as f:
        tp_indices = json.load(f)
tp_target = tp_metadata['target_column']

# For Mushroom all categorical feature columns are listed in exact saved order.
# For mixed-type datasets we only proceed if the modeling CSV already contains
# precisely the selected feature columns. Otherwise we need the project settings.
tp_features = [c for c in tp_raw.columns if c != tp_target]
tp_expected_count = int(tp_metadata['feature_count'])
tp_categorical_cols = tp_metadata.get('categorical_feature_columns', [])
if len(tp_categorical_cols) == tp_expected_count:
    tp_features = list(tp_categorical_cols)
if len(tp_features) != tp_expected_count:
    raise ValueError(
        f'TabPFN expects {tp_expected_count} features but found {len(tp_features)}. '
        'Use the exact feature column order from the AVISTA project configuration.')
if len(set(tp_features)) != len(tp_features) or any(c not in tp_raw.columns for c in tp_features):
    raise ValueError('TabPFN feature names missing/duplicated in raw dataset')

# Important: AVISTA uses .loc on split indices, not a fresh random split.


def tp_subset(split_key):
    idx = tp_indices[split_key]
    return tp_raw.loc[idx, tp_features].copy(), tp_raw.loc[idx, tp_target].copy()


tp_X_train, tp_target_train = tp_subset('train_index')
tp_X_val, tp_target_val = tp_subset('validation_index')
tp_X_test, tp_target_test = tp_subset('test_index')
tp_y_train = np.asarray(data['y_train_encoded'], dtype=np.int64)
tp_y_val = np.asarray(data['y_val_encoded'], dtype=np.int64)
tp_y_test = np.asarray(data['y_test_encoded'], dtype=np.int64)
for label, yraw, yencoded, yexport in (
    ('train', tp_target_train, tp_y_train, data['y_train']),
    ('validation', tp_target_val, tp_y_val, data['y_val']),
        ('test', tp_target_test, tp_y_test, data['y_test'])):
    if len(yraw) != len(yencoded) or not np.array_equal(
            yraw.astype(str).to_numpy(), np.asarray(yexport).astype(str)):
        raise ValueError(
            f'{label}: original raw targets do not match AVISTA saved split order')

tp_checkpoint = Path(tp_config['model_path'])
if not tp_checkpoint.is_file():
    raise FileNotFoundError(
        f'TabPFN 2.5 checkpoint not found at {tp_checkpoint}; '
        'locate the official cached checkpoint and update the recorded path.')


def tp_sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


tp_sha = tp_sha256(tp_checkpoint)
assert tp_sha == tp_metadata['tabpfn_checkpoint_sha256'], 'Checkpoint SHA256 differs from AVISTA'
if tabpfn.__version__ != tp_metadata['tabpfn_version']:
    print('WARNING: TabPFN version mismatch:', tabpfn.__version__,
          'vs. AVISTA', tp_metadata['tabpfn_version'])
tp_device = 'cuda' if torch.cuda.is_available() else 'cpu'
if tp_device != tp_metadata['selected_device']:
    print('WARNING: device differs from AVISTA:', tp_device,
          'vs.', tp_metadata['selected_device'])
tp_categories = list(tp_config['categorical_features_indices'])
assert all(0 <= i < len(tp_features) for i in tp_categories)
assert [tp_features[i] for i in tp_categories] == tp_categorical_cols, (
    'Categorical feature order differs from AVISTA'
)


def tp_new_model():
    return TabPFNClassifier(
        n_estimators=int(tp_config['n_estimators']),
        categorical_features_indices=tp_categories,
        model_path=str(tp_checkpoint),
        device=tp_device,
        random_state=tp_seed,
    )

# AVISTA uses prediction batches of exactly 500 rows; align class columns.


def tp_metrics(model, X, y):
    pieces = [np.asarray(model.predict_proba(X.iloc[i:i+500]))
              for i in range(0, len(X), 500)]
    raw_probs = np.vstack(pieces)
    probs = np.zeros((len(X), len(np.unique(tp_y_train))), dtype=float)
    for col, class_id in enumerate(model.classes_):
        probs[:, int(class_id)] = raw_probs[:, col]
    pred = probs.argmax(axis=1)
    metrics = {
        'accuracy': accuracy_score(y, pred),
        'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y, pred, average='weighted', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
        'macro_precision': precision_score(y, pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y, pred, average='macro', zero_division=0)
    }
    try:
        metrics['roc_auc'] = (roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2
                              else roc_auc_score(y, probs, multi_class='ovr', average='macro'))
    except ValueError:
        metrics['roc_auc'] = np.nan
    return metrics, pred


# In the three small example datasets, the 50,000-row TabPFN limit is not hit.
# Fail explicitly if it is: otherwise the effective training rows would differ.
if len(tp_X_train) > 50_000:
    raise NotImplementedError(
        'Dataset exceeds AVISTA TabPFN 50k limit; implement its deterministic stratified cap.')

print('TabPFN | dataset:', PROJECT_DIR.name, '| version:', tabpfn.__version__,
      '| device:', tp_device, '| features:', len(tp_features),
      '| seed:', tp_seed, '| estimators:', tp_config['n_estimators'])
tp_model = tp_new_model()
tp_model.fit(tp_X_train, tp_y_train)
tp_rows = []
for split, (X, y) in {'Validation': (tp_X_val, tp_y_val),
                      'Test': (tp_X_test, tp_y_test)}.items():
    measured, _ = tp_metrics(tp_model, X, y)
    reference_metrics = tp_reference.get(split.lower() + '_metrics', {})
    for metric, value in measured.items():
        av = reference_metrics.get(metric, np.nan)
        tp_rows.append({'Dataset': PROJECT_DIR.name, 'Model': 'TabPFN 2.5',
                        'Split': split, 'Metric': metric, 'AVISTA': av,
                        'Python': value, 'Absolute Difference': abs(value - av)})
tp_comparison_df = pd.DataFrame(tp_rows)
display(tp_comparison_df.round(6))
tp_comparison_df.to_csv(
    tp_output / 'TabPFN_reproducibility_results.csv', index=False)

# Here five-fold CV is on the same raw rows with AVISTA's same StratifiedKFold.
# Unlike the generic processed-feature CV, no external fold preprocessing is used.
TP_RUN_CV = True
if TP_RUN_CV:
    splitter = StratifiedKFold(
        n_splits=tp_folds, shuffle=True, random_state=tp_seed)
    tp_fold_rows = []
    for fold, (train_pos, validation_pos) in enumerate(
        splitter.split(np.zeros(len(tp_y_train)), tp_y_train), start=1
    ):
        if len(train_pos) > 50_000:
            raise NotImplementedError(
                'Fold training size exceeds AVISTA TabPFN 50k cap')
        reference_model = tp_new_model()
        reference_model.fit(
            tp_X_train.iloc[train_pos].copy(), tp_y_train[train_pos])
        measured, _ = tp_metrics(reference_model, tp_X_train.iloc[validation_pos].copy(),
                                 tp_y_train[validation_pos])
        tp_fold_rows.append({'Fold': fold, **measured})
        print('TabPFN CV fold', fold, 'of', tp_folds,
              '| Macro-F1:', round(measured['macro_f1'], 6))
        del reference_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    tp_folds_df = pd.DataFrame(tp_fold_rows)
    tp_folds_df.to_csv(tp_output / 'TabPFN_cv_fold_results.csv', index=False)
    tp_cv_rows = []
    for metric in ('accuracy', 'macro_f1', 'weighted_f1', 'balanced_accuracy', 'roc_auc'):
        av = tp_reference.get('cv_summary', {}).get(metric, {})
        tp_cv_rows.append({'Metric': metric, 'AVISTA Mean': av.get('mean', np.nan),
                           'Python Mean': tp_folds_df[metric].mean(),
                           'AVISTA Std': av.get('std', np.nan),
                           'Python Std': tp_folds_df[metric].std(ddof=0)})
    tp_cv_comparison_df = pd.DataFrame(tp_cv_rows)
    display(tp_cv_comparison_df.round(6))
    tp_cv_comparison_df.to_csv(
        tp_output / 'TabPFN_cv_comparison.csv', index=False)
print('TabPFN outputs saved to:', tp_output)

TabPFN | dataset: national-poll2 | version: 9.2.0 | device: cuda | features: 13 | seed: 42 | estimators: 8


,Dataset,Model,Split,Metric,AVISTA,Python,Absolute Difference
0,national-poll2,TabPFN 2.5,Validation,accuracy,0.527778,0.527778,0.000000
1,national-poll2,TabPFN 2.5,Validation,macro_f1,0.230303,0.230303,0.000000
2,national-poll2,TabPFN 2.5,Validation,weighted_f1,0.364646,0.364646,0.000000
3,national-poll2,TabPFN 2.5,Validation,balanced_accuracy,0.333333,0.333333,0.000000
4,national-poll2,TabPFN 2.5,Validation,macro_precision,0.175926,0.175926,0.000000
5,national-poll2,TabPFN 2.5,Validation,macro_recall,0.333333,0.333333,0.000000
6,national-poll2,TabPFN 2.5,Validation,roc_auc,0.592084,0.591284,0.000800
7,national-poll2,TabPFN 2.5,Test,accuracy,0.517483,0.517483,0.000000
8,national-poll2,TabPFN 2.5,Test,macro_f1,0.227343,0.227343,0.000000
9,national-poll2,TabPFN 2.5,Test,weighted_f1,0.352937,0.352937,0.000000


TabPFN CV fold 1 of 5 | Macro-F1: 0.22807
TabPFN CV fold 2 of 5 | Macro-F1: 0.22807
TabPFN CV fold 3 of 5 | Macro-F1: 0.22807
TabPFN CV fold 4 of 5 | Macro-F1: 0.22807
TabPFN CV fold 5 of 5 | Macro-F1: 0.229581


,Metric,AVISTA Mean,Python Mean,AVISTA Std,Python Std
0,accuracy,0.521051,0.521051,0.002101,0.002101
1,macro_f1,0.228372,0.228372,0.000604,0.000604
2,weighted_f1,0.356984,0.356984,0.002390,0.002390
3,balanced_accuracy,0.333333,0.333333,0.000000,0.000000
4,roc_auc,0.509765,0.510477,0.032921,0.033849


TabPFN outputs saved to: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproduciablity_python
